In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from index.inverted_index import Index
from index.documents import load_dataset, tokenizer

msmarco = load_dataset("msmarco-passage/dev/judged")

In [3]:
import locale, sys
print(sys.flags.utf8_mode)                 # should be 1
print(locale.getpreferredencoding(False))  # should be UTF-8

1
utf-8


In [4]:
NUM_DOCS = -1
K = 10

In [ ]:
from time import perf_counter

msmarco_index = Index()
msmarco_index.clear_index()

load_time = perf_counter()
msmarco_index.build_index(msmarco, tokenizer, max_num_docs=NUM_DOCS)
load_time_end = perf_counter()

print(f"Indexing time: {load_time_end - load_time:.2f} seconds")

compression_start = perf_counter()
msmarco_index.compress_index(method="gamma")
compression_end = perf_counter()

print(f"Compression time: {compression_end - compression_start:.2f} seconds")

save_start = perf_counter()
msmarco_index.save_index("msmarco_index.bin")
save_end = perf_counter()

print(f"Save time: {save_end - save_start:.2f} seconds")

decompression_start = perf_counter()
msmarco_index.decompress_index()
decompression_end = perf_counter()

print(f"Decompression time: {decompression_end - decompression_start:.2f} seconds")

loaded_index = Index()
loaded_index.clear_index()

load_start = perf_counter()
loaded_index.load_index("msmarco_index.bin")
load_end = perf_counter()

print(f"Load time with gamma decompression: {load_end - load_start:.2f} seconds")

del loaded_index
del msmarco_index

Processing:   23.1% [2039000/8841823] | Current docid: 2039000             

In [ ]:
def check_correctness(original_index, loaded_index):
    for term in original_index.index.keys():
        original_postings = original_index.index[term]
        loaded_postings = loaded_index.index.get(term, [])
        
        if original_postings != loaded_postings:
            print(f"Mismatch for term '{term}':")
            print(f"Original: {original_postings}")
            print(f"Loaded: {loaded_postings}")
            return False
    return True

In [ ]:
msmarco_index = Index()
msmarco_index.build_index(msmarco, tokenizer, max_num_docs=NUM_DOCS)

## Sparse tf_idf Matrix

In [ ]:
from scipy.sparse import csr_matrix
import numpy as np
import pandas as pd

def create_sparse_tf_idf_matrix(index, dtype=np.float32):
    terms = list(index.get_terms())
    term_to_col = {t:i for i,t in enumerate(terms)}
    doc_ids = index.get_doc_ids()
    doc_to_row = {did:i for i,did in enumerate(doc_ids)}

    rows, cols, data = [], [], []
    for term, posting in index.index.items():
        idf = index.get_inverse_document_frequency(term)
        col = term_to_col[term]
        for doc_id, tf in posting:
            rows.append(doc_to_row[doc_id])
            cols.append(col)
            data.append(np.log10(1+tf) * idf)

    data = np.array(data, dtype=dtype)
    mat = csr_matrix((data, (rows, cols)), shape=(len(doc_ids), len(terms)), dtype=dtype)
    return mat, doc_ids, terms

In [ ]:
mat, doc_ids, terms = create_sparse_tf_idf_matrix(msmarco_index)
df = pd.DataFrame.sparse.from_spmatrix(mat, index=doc_ids, columns=terms)

In [ ]:
# term_index = {term: idx for idx, term in enumerate(terms)}
# # Transform each document into a set of termIDs

# doc_set = {doc_id: set(term_index[term] for term in msmarco.docs_iter()[1] f ) for doc_id in doc_ids}

In [ ]:
doc_id_term_id = msmarco_index.compute_docId_termId(msmarco, tokenizer, max_num_docs=NUM_DOCS)

In [ ]:
# Reorder the collection according to the size of document set (reverse order)
# doc_ids = sorted(doc_id_term_id, key=lambda x: len(doc_id_term_id.get(x)), reverse=True)
doc_id_term_id = {
    doc_id: doc_id_term_id[doc_id] 
    for doc_id in sorted(doc_id_term_id, key=lambda x: len(doc_id_term_id.get(x)), reverse=True)
}
# Scan linearly the collection of document to clustering them using the Jaccard distance = 1 - JaccardSimilarity

print(doc_id_term_id)

In [ ]:
# Stream_cluster(SortedCollection, Radius)
# C = ∅
# for each d in SortedCollection
# Dist_c = Min (JaccardDistance(c, d), for each medoid c in C)
# if (Dist_c < Radius) then
# add d to cluster c
# else
# make d a new medoid, and add it to C: C = C ∪ d
# return C

In [ ]:
from sklearn.metrics import jaccard_score

# def jaccard_distance(x, y):
#     intersection = np.sum(np.minimum(x, y))
#     union = np.sum(np.maximum(x, y))
#     return 1 - intersection / union if union > 0 else 0

# def stream_cluster(SortedCollection, Radius, k):
#     C = []
#     Clusters = {}

#     for d in SortedCollection:
#             if C:
#                 # Calcoli la lista delle distanze una volta sola
#                 distances = [1 - jaccard_score(list(c), list(d), average='binary') for c in C]
                
#                 # Estrai indice e valore minimo senza ricalcolare
#                 cluster = np.argmin(distances)
#                 dist_c = distances[cluster]
#             else:
#                 cluster, dist_c = 1, 0

#     if dist_c < Radius:
#         Clusters[cluster].append(d)
#     else:
#         # make d a new medoid, and add it to C
#         C.append(d)
#     return C

from collections import defaultdict
from sklearn.metrics import jaccard_score
import numpy as np

def stream_cluster(SortedCollection, Radius, k):
    C = []                      # Conterrà le liste di term_id dei medoidi
    Clusters = defaultdict(list) # Conterrà gli ID (chiavi) dei documenti suddivisi per cluster

    # Iteriamo usando .items() per avere sia l'ID del documento che la sua lista di termini
    for doc_id, doc_terms in SortedCollection.items():
        
        set_d = set(doc_terms)
        # Se C è vuoto, il primo documento diventa il primo medoide
        if not C:
            Clusters[0].append(doc_id) # Salva l'ID nel cluster 0
            C.append(doc_id)
            continue

        # Calcolo delle distanze usando doc_terms
        distances = []
        for c in C:
            # print(SortedCollection)
            centroid_set = set(SortedCollection.get(c))
            # set_c = set(c) # Recuperiamo i termini del medoide tramite il suo ID 'c'
            
            # Formula di Jaccard: 1 - (Intersezione / Unione)
            intersection = len(centroid_set.intersection(set_d))
            union = len(centroid_set.union(set_d))
            
            jaccard_dist = 1 - (intersection / union) if union > 0 else 1
            distances.append(jaccard_dist)
        
        # print(distances)
        cluster = np.argmin(distances)
        dist_c = distances[cluster]

        if dist_c < Radius or len(C) > k: #! fixes number of clusters but still very slow
            # Assegna l'ID del documento al cluster esistente
            Clusters[cluster].append(doc_id)
        else:
            # Crea un nuovo medoide con i termini del documento corrente
            C.append(doc_id)
            nuovo_indice_cluster = len(C) - 1
            Clusters[nuovo_indice_cluster].append(doc_id)
            
    return C, Clusters

In [ ]:
C, Clusters = stream_cluster(doc_id_term_id, 0.8, 1)
print(f"Number of clusters: {len(C)}")

## Kmeans + TCP

In [ ]:
# from sklearn.cluster import KMeans, DBSCAN
# from sklearn.preprocessing import normalize

# def jaccard_distance(x, y):
#     intersection = np.sum(np.minimum(x, y))
#     union = np.sum(np.maximum(x, y))
#     return 1 - intersection / union if union > 0 else 0

# clustering = DBSCAN(metric=jaccard_distance, eps=0.5, min_samples=2)

# # norm_df = normalize(df, norm="l2") #! cosa fa , è fondamentele se non la metti clusteing fa schifo 

# # kmeans = KMeans(n_clusters=K, random_state=0) #! use jaccard distance instead of euclidean

# clusters = clustering.fit(doc_id_term_id.values())
# print("Cluster centers:", clustering.cluster_centers_)

# document_clusters = clustering.labels_
# print("Document clusters:", document_clusters)

## Medoids

In [ ]:
# medoids = []

# for i in range(kmeans.n_clusters):
#     cluster_docs = np.flatnonzero(document_clusters == i)
#     cluster_vectors = norm_df[cluster_docs]
#     medoid_idx = np.argmin(np.linalg.norm(cluster_vectors - kmeans.cluster_centers_[i], axis=1))
#     medoids.append(cluster_docs[medoid_idx])



# medoid_vectors = norm_df[medoids]
# nonzero_cols = medoid_vectors.getnnz(axis=0) > 0
# medoid_vectors = medoid_vectors[:, nonzero_cols].toarray()


In [ ]:
from sklearn.metrics import pairwise_distances
import numpy as np

def calculate_distance_matrix(medoid_vectors, method="euclidean"):
    
    dist = np.zeros((len(medoid_vectors), len(medoid_vectors)))

    for i, v1 in enumerate(medoid_vectors):
        set_v1 = set(v1)

        for j, v2 in enumerate(medoid_vectors):

            if(i == j):
                continue
            
            set_v2 = set(v2)            
            # Formula di Jaccard: 1 - (Intersezione / Unione)
            intersection = len(set_v1.intersection(set_v2))
            union = len(set_v1.union(set_v2))
            
            jaccard_dist = 1 - (intersection / union) if union > 0 else 1
            dist[i][j] = jaccard_dist
        
    return dist
    
    # if method == "euclidean":
    #     return pairwise_distances(medoid_vectors, metric="euclidean")
    # elif method == "cosine":
    #     return pairwise_distances(medoid_vectors, metric="cosine")
    # elif method == "jaccard":
    #     # Jaccard in sklearn works on boolean/binary features
    #     return pairwise_distances(medoid_vectors, metric="jaccard")
    # else:
    #     raise ValueError(f"Unknown method: {method}")

In [ ]:
medoid_vectors = [doc_id_term_id.get(c) for c in C]

In [ ]:
dist_matrix = calculate_distance_matrix(medoid_vectors, method="jaccard")

In [ ]:
import networkx as nx
tsp = nx.approximation.traveling_salesman_problem

tsp_graph = nx.from_numpy_array(dist_matrix)
tsp_path = tsp(tsp_graph, cycle=False)
print("TSP path (medoid indices):", tsp_path)

In [ ]:
# def reorder_documents_by_tsp(document_clusters, medoids, tsp_path):
#     """
#     Reorder documents by cluster order induced by TSP over medoids.
#     Inside each cluster, keep documents in their original order (row order).

#     Args:
#         document_clusters: array-like of shape (n_docs,), cluster label per doc.
#         medoids: list of medoid doc indices (not required for ordering here, kept for API compatibility).
#         tsp_path: list of cluster ids in the order returned by TSP.

#     Returns:
#         ordered_doc_ids: list of document row indices in final order.
#         cluster_to_docs: dict {cluster_id: [doc_ids in original order]}.
#     """
#     document_clusters = np.asarray(document_clusters)

#     # Build cluster -> docs preserving natural/original order
#     cluster_to_docs = {
#         c: np.flatnonzero(document_clusters == c).tolist()
#         for c in range(len(medoids))
#     }

#     # Concatenate clusters following TSP order
#     ordered_doc_ids = []
#     for c in tsp_path:
#         ordered_doc_ids.extend(cluster_to_docs.get(c, []))

#     return ordered_doc_ids, cluster_to_docs

In [ ]:
# ordered_doc_ids, cluster_docs = reorder_documents_by_tsp(Clusters, C, tsp_path)

In [ ]:
def reorder_documents_by_tsp_new(tsp_path, cluster, C):
    tot = []
    for i in tsp_path:
        docs = cluster.get(i)
        for d in docs:  
            tot.append(d)
    return tot

reord_docids = reorder_documents_by_tsp_new(tsp_path,Clusters,C)
print(min(reord_docids))

In [ ]:
from itertools import islice

docs = list(islice(msmarco.docs_iter(), NUM_DOCS))

class OrderedDataset:
    def __init__(self, docs, order):
        self.docs = docs
        self.order = order
    def docs_iter(self):
        for i in self.order:
            yield self.docs[i-1]

ordered_dataset = OrderedDataset(docs, reord_docids)
reordered_inverted_index = Index()
reordered_inverted_index.build_index(
    ordered_dataset,
    tokenizer,
    max_num_docs=-1, #! -1 since reordered docs are already limited to 10001 in the original index
    d_gap=True
)

stat_r = reordered_inverted_index.get_approx_size()

msmarco_index = Index()
msmarco_index.build_index(msmarco, tokenizer, max_num_docs=1000, d_gap=True)

stat_m = msmarco_index.get_approx_size()
print(f"Original index size: {stat_m} bytes")
print(f"Reordered index size: {stat_r} bytes")

ind_1 = reordered_inverted_index.index
ind_2 = msmarco_index.index

print("Msmarco index size:", len(msmarco_index.index))

In [ ]:
reordered_inverted_index.compress_index("gamma")
reordered_inverted_index.save_index(file_path="reord.bin")
msmarco_index.compress_index("gamma")
msmarco_index.save_index(file_path="not_reord.bin")